# Simple XRK Data Analysis with Libxrk

This notebook demonstrates using libxrk to do easy analysis of AIM(tm) datasets without needing any AIM software installed locally.

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions (includes show_fig for JupyterLite compatibility)
from motorsports_data_notebook import (
    show_fig, get_best_lap, compute_start_line, plot_lap_gps,
    gps_to_local_xy, compute_curvature, compute_lap_distance,
    identify_corners, identify_corners_from_curvature
)

In [ ]:
log = aim_xrk("CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")

In [ ]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()

# Add derived columns
channels['speed_kmh'] = channels['GPS Speed'] * 3.6

In [ ]:
# Load laps and compute lap times
laps = log.laps.to_pandas()
laps['lap_time'] = pd.to_timedelta(laps['end_time'] - laps['start_time'], unit='ms')

# Compute distance_m for each lap and add to channels
# Distance resets at the start of each lap
channels['distance_m'] = 0.0

for idx, lap in laps.iterrows():
    lap_mask = (channels['timecodes'] >= lap['start_time']) & (channels['timecodes'] <= lap['end_time'])
    lap_indices = channels.index[lap_mask]
    
    if len(lap_indices) > 0:
        lap_timecodes = channels.loc[lap_indices, 'timecodes']
        lap_speed = channels.loc[lap_indices, 'GPS Speed']
        distance_values = compute_lap_distance(lap_timecodes.values, lap_speed.values)
        channels.loc[lap_indices, 'distance_m'] = distance_values

laps.style.format({'lap_time': lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"})

In [ ]:
channels.columns

In [ ]:
# Best lap extraction
best_lap = get_best_lap(laps)
start_ts = best_lap['start_time']
end_ts = best_lap['end_time']
# Use < for end_ts to exclude the first sample of the next lap (where distance resets to 0)
lap_channels = channels.query(f'timecodes >= @start_ts and timecodes < @end_ts').copy()

In [ ]:

# plot speed on GPS map
fig = plot_lap_gps(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    color_channels=[
        (lap_channels['speed_kmh'], 'Speed (km/h)', 'Viridis')
    ],
    title='Speed'
)
show_fig(fig)

In [ ]:

# Plot with multiple color channels
fig = plot_lap_gps(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    color_channels=[
        (lap_channels['BrakePress'], 'BrakePres', 'Reds'),
        (lap_channels['PPS'], 'Throttle', 'Greens')
    ],
    title='Accelerator and Brake Pressure on Best Lap'
)
show_fig(fig)

In [ ]:
# Tire Thermography - Best Lap
# Extract FL, FR, RL, RR tire temperature channels
# Ch1 = leftmost (outside for FL/RL, inside for FR/RR), Ch8 = rightmost (inside for FL/RL, outside for FR/RR)
fl_channels = ['FL_Ch1', 'FL_Ch2', 'FL_Ch3', 'FL_Ch4', 'FL_Ch5', 'FL_Ch6', 'FL_Ch7', 'FL_Ch8']
fr_channels = ['FR_Ch1', 'FR_Ch2', 'FR_Ch3', 'FR_Ch4', 'FR_Ch5', 'FR_Ch6', 'FR_Ch7', 'FR_Ch8']
rl_channels = ['RL_Ch1', 'RL_Ch2', 'RL_Ch3', 'RL_Ch4', 'RL_Ch5', 'RL_Ch6', 'RL_Ch7', 'RL_Ch8']
rr_channels = ['RR_Ch1', 'RR_Ch2', 'RR_Ch3', 'RR_Ch4', 'RR_Ch5', 'RR_Ch6', 'RR_Ch7', 'RR_Ch8']

fl_temps = lap_channels[fl_channels].values.T  # Shape: (8 channels, n_samples)
fr_temps = lap_channels[fr_channels].values.T
rl_temps = lap_channels[rl_channels].values.T
rr_temps = lap_channels[rr_channels].values.T

# Use pre-computed distance_m from lap_channels
distance_m = lap_channels['distance_m']

# Calculate Sum of G (Euclidean sum of lateral and inline accelerations)
sum_of_g = np.sqrt(lap_channels['LateralAcc']**2 + lap_channels['InlineAcc']**2)

# Get color scale range across all tires for consistent coloring
vmin = min(fl_temps.min(), fr_temps.min(), rl_temps.min(), rr_temps.min())
vmax = max(fl_temps.max(), fr_temps.max(), rl_temps.max(), rr_temps.max())

# Create subplots with shared x-axis (6 rows: 4 tire heatmaps + speed/G plot + inputs plot)
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=('Front Left (Outside at top)', 'Front Right (Outside at bottom)', 
                    'Rear Left (Outside at top)', 'Rear Right (Outside at bottom)', 
                    'Speed & Sum of G', 'Driver Inputs'),
    row_heights=[0.17, 0.17, 0.17, 0.17, 0.16, 0.16],
    specs=[[{}], [{}], [{}], [{}], [{"secondary_y": True}], [{"secondary_y": True}]]
)

# Y-axis labels (just channel numbers, no prefix)
y_labels = ['1', '2', '3', '4', '5', '6', '7', '8']

# Front Left heatmap - FL Ch1 (outside/left) at top
fig.add_trace(
    go.Heatmap(
        z=fl_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        showscale=False
    ),
    row=1, col=1
)
fig.update_yaxes(autorange='reversed', row=1, col=1)

# Front Right heatmap - FR Ch8 (outside/right) at bottom
fig.add_trace(
    go.Heatmap(
        z=fr_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        showscale=False
    ),
    row=2, col=1
)
fig.update_yaxes(autorange='reversed', row=2, col=1)

# Rear Left heatmap - RL Ch1 (outside/left) at top
fig.add_trace(
    go.Heatmap(
        z=rl_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        showscale=False
    ),
    row=3, col=1
)
fig.update_yaxes(autorange='reversed', row=3, col=1)

# Rear Right heatmap - RR Ch8 (outside/right) at bottom
fig.add_trace(
    go.Heatmap(
        z=rr_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        colorbar=dict(title='Temp (°C)')
    ),
    row=4, col=1
)
fig.update_yaxes(autorange='reversed', row=4, col=1)

# Speed line plot at row 5 (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels['speed_kmh'].values,
        mode='lines',
        name='Speed',
        line=dict(color='black', width=1)
    ),
    row=5, col=1, secondary_y=False
)

# Sum of G line plot (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=sum_of_g.values,
        mode='lines',
        name='Sum of G',
        line=dict(color='red', width=1)
    ),
    row=5, col=1, secondary_y=True
)

# Driver Inputs subplot (row 6)
# Brake Pressure - red (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels['BrakePress'].values,
        mode='lines',
        name='Brake',
        line=dict(color='red', width=1)
    ),
    row=6, col=1, secondary_y=False
)

# Throttle (PPS) - green (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels['PPS'].values,
        mode='lines',
        name='Throttle',
        line=dict(color='green', width=1)
    ),
    row=6, col=1, secondary_y=False
)

# Steering Angle - black (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels['SteerAngle'].values,
        mode='lines',
        name='Steering',
        line=dict(color='black', width=1)
    ),
    row=6, col=1, secondary_y=True
)

fig.update_layout(
    title='Tire Temperatures - Best Lap',
    xaxis6_title='Distance (m)',
    yaxis_title='FL',
    yaxis2_title='FR',
    yaxis3_title='RL',
    yaxis4_title='RR',
    width=900,
    height=900,
    showlegend=False
)

# Set y-axis titles for the speed/G subplot (row 5)
fig.update_yaxes(title_text='km/h', row=5, col=1, secondary_y=False)
fig.update_yaxes(title_text='G', row=5, col=1, secondary_y=True)

# Set y-axis titles for the driver inputs subplot (row 6)
fig.update_yaxes(title_text='%', row=6, col=1, secondary_y=False)
fig.update_yaxes(title_text='deg', row=6, col=1, secondary_y=True)

# Hide tick labels on y-axes for heatmaps (keep only the axis title)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_yaxes(showticklabels=False, row=3, col=1)
fig.update_yaxes(showticklabels=False, row=4, col=1)

show_fig(fig)

# Lap Segmentation & Consistency Analysis

This section analyzes braking and acceleration consistency by:
1. Computing track curvature from GPS data to identify corners
2. Detecting braking and acceleration zones from telemetry
3. Creating fixed segment boundaries for the track
4. Computing statistics across all laps to show variation

In [ ]:
# Step 2: Identify corners directly from GPS coordinates
# identify_corners handles GPS->XY conversion, curvature computation, and corner detection
corners = identify_corners(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    threshold=0.006,  # Tuned for GPS noise - ~167m radius threshold
    min_corner_length=15,  # Reduced to catch shorter corners
    min_gap=80  # Merge same-direction corners within 80m
)

print(f"Found {len(corners)} corners:")
for c in corners:
    print(f"  {c.name} ({c.direction}): {c.start_dist:.0f}m - {c.end_dist:.0f}m (apex at {c.apex_dist:.0f}m, radius ~{c.radius:.0f}m)")

In [ ]:
# Visualize corners on GPS map with markers
fig = go.Figure()

# Plot track colored by speed
fig.add_trace(go.Scattermapbox(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    mode='markers',
    marker=dict(
        size=5,
        color=lap_channels['speed_kmh'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Speed (km/h)')
    ),
    name='Track'
))

# Add corner apex markers
for corner in corners:
    apex_idx = corner.apex_idx
    fig.add_trace(go.Scattermapbox(
        lat=[lap_channels['GPS Latitude'].iloc[apex_idx]],
        lon=[lap_channels['GPS Longitude'].iloc[apex_idx]],
        mode='markers+text',
        marker=dict(size=15, color='red'),
        text=[corner.name],
        textposition='top right',
        textfont=dict(size=12, color='red'),
        name=corner.name
    ))

fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(
            lat=lap_channels['GPS Latitude'].mean(),
            lon=lap_channels['GPS Longitude'].mean()
        ),
        zoom=14
    ),
    title='Detected Corners',
    showlegend=False,
    width=800,
    height=600
)

show_fig(fig)

In [ ]:
# Step 3: Identify braking and acceleration zones averaged over top laps
# Reload helpers to pick up new functions
import importlib
import motorsports_data_notebook.helpers
importlib.reload(motorsports_data_notebook.helpers)
from motorsports_data_notebook.helpers import (
    identify_zones_single_lap, average_zones_across_laps, merge_accel_zones_by_time
)

# Get laps within 103% of best lap time
valid_laps = laps[laps['lap_time'] > pd.Timedelta(0)].copy()
best_lap_time = valid_laps['lap_time'].min()
threshold_time = best_lap_time * 1.03
top_laps = valid_laps[valid_laps['lap_time'] <= threshold_time]

print(f"Best lap time: {best_lap_time}")
print(f"103% threshold: {threshold_time}")
print(f"Using {len(top_laps)} laps within 103% of best (out of {len(valid_laps)} valid laps) for zone averaging")

# Collect zones from each top lap
all_braking_zones = []
all_accel_zones = []

for idx, lap in top_laps.iterrows():
    lap_start = lap['start_time']
    lap_end = lap['end_time']
    lap_data = channels.query(f'timecodes >= @lap_start and timecodes <= @lap_end').copy()
    
    if len(lap_data) < 10:
        continue
    
    # Use pre-computed distance_m from channels
    braking, accel = identify_zones_single_lap(
        lap_data['distance_m'].values,
        lap_data['BrakePress'].values,
        lap_data['PPS'].values,
        lap_data['GPS Speed'].values  # Speed in m/s for time-based gear change detection
    )
    all_braking_zones.append(braking)
    all_accel_zones.append(accel)

# Average zones across laps (use 50% threshold - at least half of laps must agree)
braking_zones, accel_zones = average_zones_across_laps(
    all_braking_zones, all_accel_zones, 
    track_length=lap_channels['distance_m'].max(),
    resolution=1.0,
    threshold=0.5
)

# Post-process: merge acceleration zones separated by short time gaps (gear changes)
accel_zones = merge_accel_zones_by_time(
    accel_zones, braking_zones,
    lap_channels['distance_m'].values,
    lap_channels['GPS Speed'].values,
    max_gap_time=1.5  # 1.5 seconds to bridge gear changes
)

print(f"Found {len(braking_zones)} braking zones and {len(accel_zones)} acceleration zones (averaged over {len(top_laps)} laps)")

In [ ]:
# Debug: Investigate zones between Turn 4 (exit ~2068m) and Turn 6 (braking ~2686m)
print("=== Acceleration zones in T4-T6 region (2000m - 2900m) ===")
for i, (start, end) in enumerate(accel_zones):
    if 2000 < end and start < 2900:
        print(f"  Accel zone {i}: {start:.0f}m - {end:.0f}m (length: {end-start:.0f}m)")

print("\n=== Braking zones in T4-T6 region (2000m - 2900m) ===")
for i, (start, end) in enumerate(braking_zones):
    if 2000 < end and start < 2900:
        print(f"  Braking zone {i}: {start:.0f}m - {end:.0f}m (length: {end-start:.0f}m)")

print("\n=== Corner positions in T4-T6 region ===")
for c in corners:
    if 2000 < c.apex_dist < 2900:
        print(f"  {c.name}: apex at {c.apex_dist:.0f}m, start {c.start_dist:.0f}m, end {c.end_dist:.0f}m")

# Check the gaps between consecutive accel zones in this region
print("\n=== Gap analysis between consecutive accel zones (T4-T6 region) ===")
region_accel = [(s, e) for s, e in accel_zones if 2000 < e and s < 2900]
for i in range(len(region_accel) - 1):
    prev_end = region_accel[i][1]
    next_start = region_accel[i+1][0]
    gap_dist = next_start - prev_end
    
    # Find average speed in the gap
    mask = (lap_channels['distance_m'] >= prev_end) & (lap_channels['distance_m'] <= next_start)
    gap_speeds = lap_channels.loc[mask, 'GPS Speed'].values
    avg_speed = np.mean(gap_speeds) if len(gap_speeds) > 0 else 0
    gap_time = gap_dist / avg_speed if avg_speed > 0 else float('inf')
    
    # Check for braking in gap
    braking_in_gap = []
    for bz_start, bz_end in braking_zones:
        if bz_start < next_start and bz_end > prev_end:
            braking_in_gap.append((bz_start, bz_end))
    
    print(f"  Gap from {prev_end:.0f}m to {next_start:.0f}m:")
    print(f"    Distance: {gap_dist:.0f}m, Avg speed: {avg_speed:.1f} m/s, Gap time: {gap_time:.2f}s")
    if braking_in_gap:
        print(f"    BRAKING IN GAP: {braking_in_gap}")
    else:
        print(f"    No braking in gap")

In [ ]:
# Step 4: Create fixed segment definitions combining corners with braking/accel zones
from motorsports_data_notebook.helpers import TrackSegment, create_track_segments

# Create segments
track_length = lap_channels['distance_m'].iloc[-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"Created {len(segments)} track segments:")
for seg in segments:
    print(f"  [{seg.segment_type:12}] {seg.name:20} : {seg.start_dist:6.0f}m - {seg.end_dist:6.0f}m")

In [ ]:
# Visualize track segments on GPS map

fig = go.Figure()

# Create a distance-to-index mapping for GPS coordinates
distance_arr = lap_channels['distance_m'].values
lat_arr = lap_channels['GPS Latitude'].values
lon_arr = lap_channels['GPS Longitude'].values

def get_indices_for_range(start_dist, end_dist):
    """Get indices corresponding to a distance range."""
    mask = (distance_arr >= start_dist) & (distance_arr <= end_dist)
    return np.where(mask)[0]

# Color mapping for segment types
segment_colors = {
    'braking': 'red',
    'corner': 'orange',
    'acceleration': 'green'
}

# Plot base track (gray)
fig.add_trace(go.Scattermapbox(
    lat=lat_arr,
    lon=lon_arr,
    mode='lines',
    line=dict(width=3, color='lightgray'),
    name='Track',
    showlegend=True
))

# Track which segment types we've added to legend
legend_added = {'braking': False, 'corner': False, 'acceleration': False}

# Plot all segments from the segments list
for seg in segments:
    indices = get_indices_for_range(seg.start_dist, seg.end_dist)
    if len(indices) > 0:
        color = segment_colors.get(seg.segment_type, 'gray')
        show_in_legend = not legend_added[seg.segment_type]
        legend_added[seg.segment_type] = True
        
        legend_name = {
            'braking': 'Braking Zone',
            'corner': 'Corner',
            'acceleration': 'Acceleration Zone'
        }.get(seg.segment_type, seg.segment_type)
        
        fig.add_trace(go.Scattermapbox(
            lat=lat_arr[indices],
            lon=lon_arr[indices],
            mode='lines',
            line=dict(width=6, color=color),
            name=legend_name if show_in_legend else None,
            showlegend=show_in_legend,
            legendgroup=seg.segment_type
        ))

# Add corner apex markers with labels (for corner segments only)
for seg in segments:
    if seg.segment_type == 'corner' and seg.apex_dist is not None:
        # Find index closest to apex distance
        apex_idx = np.argmin(np.abs(distance_arr - seg.apex_dist))
        fig.add_trace(go.Scattermapbox(
            lat=[lat_arr[apex_idx]],
            lon=[lon_arr[apex_idx]],
            mode='markers+text',
            marker=dict(size=12, color='darkred', symbol='circle'),
            text=[seg.name],
            textposition='top right',
            textfont=dict(size=11, color='darkred'),
            name=None,
            showlegend=False
        ))

fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=np.mean(lat_arr), lon=np.mean(lon_arr)),
        zoom=14
    ),
    title='Track Segments: Braking (Red), Corner (Orange), Acceleration (Green)',
    legend=dict(
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(255,255,255,0.8)'
    ),
    width=900,
    height=700
)

show_fig(fig)

In [ ]:
# Step 5: Extract all laps and compute per-lap segment statistics
def find_braking_point(lap_data, segment):
    """Find the distance where braking starts within a segment."""
    mask = (lap_data['distance_m'] >= segment.start_dist) & \
           (lap_data['distance_m'] <= segment.end_dist)
    seg_data = lap_data[mask]
    
    if len(seg_data) == 0:
        return None
    
    # Find first point where brake > threshold
    brake_points = seg_data[seg_data['BrakePress'] > 5]
    if len(brake_points) > 0:
        return brake_points['distance_m'].iloc[0]
    return None

def find_throttle_point(lap_data, segment):
    """Find the distance where throttle application starts within a segment."""
    mask = (lap_data['distance_m'] >= segment.start_dist) & \
           (lap_data['distance_m'] <= segment.end_dist)
    seg_data = lap_data[mask]
    
    if len(seg_data) == 0:
        return None
    
    # Find first point where throttle > threshold and brake < threshold
    throttle_points = seg_data[(seg_data['PPS'] > 20) & (seg_data['BrakePress'] < 5)]
    if len(throttle_points) > 0:
        return throttle_points['distance_m'].iloc[0]
    return None

def find_min_speed(lap_data, segment):
    """Find minimum speed within a segment (for corners)."""
    mask = (lap_data['distance_m'] >= segment.start_dist) & \
           (lap_data['distance_m'] <= segment.end_dist)
    seg_data = lap_data[mask]
    
    if len(seg_data) == 0:
        return None
    
    return seg_data['speed_kmh'].min()

def compute_segment_stats_for_lap(lap_data, segments):
    """Compute statistics for each segment in a single lap."""
    stats = []
    
    for seg in segments:
        stat = {
            'segment_id': seg.id,
            'segment_name': seg.name,
            'segment_type': seg.segment_type,
            'corner_id': seg.corner_id
        }
        
        if seg.segment_type == 'braking':
            stat['braking_point'] = find_braking_point(lap_data, seg)
            # Calculate how early/late vs segment start
            if stat['braking_point'] is not None:
                stat['brake_offset'] = stat['braking_point'] - seg.start_dist
        
        elif seg.segment_type == 'corner':
            stat['min_speed'] = find_min_speed(lap_data, seg)
        
        elif seg.segment_type == 'acceleration':
            stat['throttle_point'] = find_throttle_point(lap_data, seg)
            if stat['throttle_point'] is not None:
                stat['throttle_offset'] = stat['throttle_point'] - seg.start_dist
        
        stats.append(stat)
    
    return stats

# Get all valid laps (exclude first and last which may be pit laps)
valid_laps = laps.iloc[1:-1] if len(laps) > 2 else laps
print(f"Analyzing {len(valid_laps)} laps...")

# Compute statistics for each lap
all_lap_stats = []

for idx, lap in valid_laps.iterrows():
    # Extract lap data (distance_m and speed_kmh already computed in channels)
    lap_data = channels.query(
        f'timecodes >= {lap["start_time"]} and timecodes <= {lap["end_time"]}'
    ).copy()
    
    if len(lap_data) < 10:
        continue
    
    # Compute segment stats
    lap_stats = compute_segment_stats_for_lap(lap_data, segments)
    
    for stat in lap_stats:
        stat['lap_num'] = lap['num']
        stat['lap_time'] = lap['lap_time']
    
    all_lap_stats.extend(lap_stats)

# Convert to DataFrame
stats_df = pd.DataFrame(all_lap_stats)
print(f"Computed {len(stats_df)} segment statistics across all laps")

In [ ]:
# Step 6: Visualize braking consistency
# Show braking point variation for each corner

braking_stats = stats_df[stats_df['segment_type'] == 'braking'].dropna(subset=['braking_point'])

if len(braking_stats) > 0:
    fig = px.box(
        braking_stats,
        x='segment_name',
        y='braking_point',
        title='Braking Point Consistency by Corner',
        labels={'braking_point': 'Braking Point (m)', 'segment_name': 'Corner'}
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No braking data available")

In [ ]:
# Visualize corner minimum speed consistency
corner_stats = stats_df[stats_df['segment_type'] == 'corner'].dropna(subset=['min_speed'])

if len(corner_stats) > 0:
    fig = px.box(
        corner_stats,
        x='segment_name',
        y='min_speed',
        title='Minimum Corner Speed Consistency',
        labels={'min_speed': 'Min Speed (km/h)', 'segment_name': 'Corner'}
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No corner speed data available")

In [ ]:
# Visualize throttle application consistency
accel_stats = stats_df[stats_df['segment_type'] == 'acceleration'].dropna(subset=['throttle_point'])

if len(accel_stats) > 0:
    fig = px.box(
        accel_stats,
        x='segment_name',
        y='throttle_point',
        title='Throttle Application Point Consistency',
        labels={'throttle_point': 'Throttle Point (m)', 'segment_name': 'Corner Exit'}
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No throttle data available")

In [ ]:
# Summary statistics table
def compute_summary_stats(stats_df):
    """Compute summary statistics for each segment across all laps."""
    summary = []
    
    # Braking segments
    for seg_name in stats_df[stats_df['segment_type'] == 'braking']['segment_name'].unique():
        seg_data = stats_df[(stats_df['segment_name'] == seg_name) & 
                           stats_df['braking_point'].notna()]
        if len(seg_data) > 0:
            summary.append({
                'Segment': seg_name,
                'Type': 'Braking',
                'Metric': 'Braking Point (m)',
                'Mean': seg_data['braking_point'].mean(),
                'Std': seg_data['braking_point'].std(),
                'Min': seg_data['braking_point'].min(),
                'Max': seg_data['braking_point'].max(),
                'Range': seg_data['braking_point'].max() - seg_data['braking_point'].min(),
                'N': len(seg_data)
            })
    
    # Corner segments
    for seg_name in stats_df[stats_df['segment_type'] == 'corner']['segment_name'].unique():
        seg_data = stats_df[(stats_df['segment_name'] == seg_name) & 
                           stats_df['min_speed'].notna()]
        if len(seg_data) > 0:
            summary.append({
                'Segment': seg_name,
                'Type': 'Corner',
                'Metric': 'Min Speed (km/h)',
                'Mean': seg_data['min_speed'].mean(),
                'Std': seg_data['min_speed'].std(),
                'Min': seg_data['min_speed'].min(),
                'Max': seg_data['min_speed'].max(),
                'Range': seg_data['min_speed'].max() - seg_data['min_speed'].min(),
                'N': len(seg_data)
            })
    
    # Acceleration segments
    for seg_name in stats_df[stats_df['segment_type'] == 'acceleration']['segment_name'].unique():
        seg_data = stats_df[(stats_df['segment_name'] == seg_name) & 
                           stats_df['throttle_point'].notna()]
        if len(seg_data) > 0:
            summary.append({
                'Segment': seg_name,
                'Type': 'Acceleration',
                'Metric': 'Throttle Point (m)',
                'Mean': seg_data['throttle_point'].mean(),
                'Std': seg_data['throttle_point'].std(),
                'Min': seg_data['throttle_point'].min(),
                'Max': seg_data['throttle_point'].max(),
                'Range': seg_data['throttle_point'].max() - seg_data['throttle_point'].min(),
                'N': len(seg_data)
            })
    
    return pd.DataFrame(summary)

summary_df = compute_summary_stats(stats_df)
summary_df.style.format({
    'Mean': '{:.1f}',
    'Std': '{:.1f}',
    'Min': '{:.1f}',
    'Max': '{:.1f}',
    'Range': '{:.1f}'
})